# Every subpopulation: error against adaptability

Whether a finetuned "improvement" is real adaptation or a relocated center.
Per question, model/variant and mode, scored over **every retained
subpopulation cell of the run** — all five countries, 687 cells where a run is
complete (639 for `d_polpos`, where 48 cells have no human answers): mean
prediction error `E = mean_i nEMD(WVS_i, LLM_i)`, dispersion
`D = median pairwise nEMD` within the survey cells and within the model's, the
adaptability ratio `AR = D_LLM / D_WVS`, and two center relocations:
`C_german = nEMD(mean_i WVS_i, mean_i LLM_i)` over the German cells and
`C_pop`, the same quantity over every retained cell.

Every population quantity has a German twin computed on the 144 German cells
alone — `e_mean_nemd_german`, `adaptability_ratio_german`, `d_*_german` — and
that twin is what the `|` tick draws. The tick is not a second run: it is the
same run scored on Germany only, so the distance between a marker and its own
tick is what this model does differently on German cells, and the distance
between the untuned variant's tick and the tuned variant's tick is what the
culture-MLLM finetuning bought on the cells it was fitted for.

A **variant** is one of the conditions a base model is evaluated under:
`base`, the untuned model with nothing attached, or a culture-finetuned
variant such as `german`. The CSV column and the run-directory path segment
are both still spelled `arm`, so `row["arm"]` in the code and "variant" in
this prose name the same thing.

`AR = 1` means the model spreads its subpopulation answers as much as the
survey's own cells do; `AR -> 0` means one answer wearing 687 names. The ideal
corner is `E -> 0`, `AR -> 1`.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = untuned `base`, solid = culture-finetuned |
| dotted rule + `\|` tick | not a run: the same run over the German cells alone |
| x | E, mean nEMD to every WVS cell |
| y | AR, model dispersion over survey dispersion |

Only fill separates the two variants, so nothing else may be drawn inside a
marker: an earlier draft put a white centre dot on every solid marker, which
at 6.5pt ate the fill and made the tuned variant read as a second hollow base.
The German companion was likewise a second *marker* in the model's pastel
tint, a third thing shaped like a variant — the supervisor read one panel as
three base runs. Both are gone: fill alone says which variant, and the German
reference is a tick and a dotted rule, a shape no series wears.

Every dashed rule on these plates is a **fixed reference, never a fit**, which
is why it never tilts: `AR = 1` on the adaptability plates, the identity
diagonal on the centers plate. They mark where a model would sit if it matched
the survey's dispersion, or sat equally far from both pools — not a trend
through the points.

A second plate puts the two relocations against each other, `x = C_german`
against `y = C_pop`, with the diagonal as the indifference rule: above it a
model's centre sits closer to the pooled German respondents than to the pooled
world, below it the reverse — where an untuned base is expected to live. Here
the German reference is the x axis itself, so no tick is drawn.

Five views per family — every question at once, then happiness, politics,
religion and trust alone — because nEMD scales with a question's answer count,
so the four panels share a y axis but never a x scale.

Recomputed from the run distributions, not the derived CSVs. Writes
`outputs/culture/population_adaptability.csv` plus
`figures/fig_population_adaptability_<view>_{ntp,fa}` and
`figures/fig_population_center_<view>_{ntp,fa}`, `.png` and `.pdf`. A run that
produced only full answers contributes its `fa` rows and no `ntp` rows, and is
scored on the cells its own mode retained. A companion notebook,
`population_structure.ipynb`, reads this CSV back in and asks a third,
orthogonal question over the same cells — not how much a model disperses but
whether it disperses along the real social axes — so it must be run after this
one, not instead of it.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell
outputs.

In [1]:
from culture.population import (
    ADAPTABILITY_TABLE,
    MODES,
    build_adaptability,
    check_adaptability,
)
from culture.population_plates import VIEWS, adaptability_plate, center_plate
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
table = build_adaptability()
check_adaptability(table)
write_csv(table, ADAPTABILITY_TABLE)
print(len(table), "rows")

113 rows


## The plates

In [3]:
files = []
for mode in MODES:
    for view in VIEWS:
        files += adaptability_plate(table, mode, view)
        files += center_plate(table, mode, view)
print(len(files), "files")

40 files


## Reading the plates

A marker low on the y axis is a model answering nearly the same way for every
subpopulation it was asked about — the compression the paper reports. A marker
near `AR = 1` disperses as much as the survey does, which is necessary for
real adaptation but not sufficient: `population_structure.ipynb` asks whether
that dispersion runs along the survey's own social divides or is noise of the
right size.

The `|` tick is the same run over the German cells alone. Read it against its
own marker, and read the two variants' ticks against each other: a tuned
variant whose tick sits left of the untuned variant's tick fits German cells
more closely than the untuned model did. A tuned variant whose *marker* moved
right while its tick moved left bought Germany at the rest of the world's
expense — the trade the centers plate then shows directly.

On the centers plate, above the diagonal is a model whose pooled answer sits
closer to the pooled German respondents than to the pooled world. An untuned
base is expected below it. A tuned variant that crossed the diagonal relocated
its centre, which is the cheap way to look adapted: the adaptability plate is
where you check whether the dispersion followed.

The p-values in the structure companion are descriptive only — pairwise
distances are not independent — and nothing here is a significance claim.